# 47. Augmentation과 입출력 해상도 실험

이 노트북은 segmentation 학습에서 augmentation과 resize가 어떤 의미를 갖는지 다룹니다.

이번 노트북의 목표는 다음과 같습니다.

- image와 mask에 같은 기하학적 변환을 적용해야 하는 이유를 이해합니다.
- image 변환과 mask 변환의 interpolation 차이를 확인합니다.
- 입력 해상도와 성능, 속도, memory의 trade-off를 정리합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from matplotlib.colors import ListedColormap

np.random.seed(4)

## 47-1. 예제 image-mask pair

In [ ]:
h, w = 96, 96
yy, xx = np.mgrid[:h, :w]
image = np.zeros((h, w, 3), dtype=np.float32) + 0.55
mask = np.zeros((h, w), dtype=np.int64)

circle = (xx - 36) ** 2 + (yy - 42) ** 2 < 18 ** 2
rect = (xx > 52) & (xx < 82) & (yy > 28) & (yy < 74)
image[circle] = [0.9, 0.25, 0.2]
image[rect] = [0.2, 0.55, 0.9]
mask[circle] = 1
mask[rect] = 2

cmap = ListedColormap(["#8b95a1", "#e53935", "#2563eb"])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(image)
axes[0].set_title("image")
axes[1].imshow(mask, cmap=cmap, vmin=0, vmax=2)
axes[1].set_title("mask")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 47-2. 같은 flip을 image와 mask에 적용하기

In [ ]:
flip_image = np.flip(image, axis=1).copy()
flip_mask = np.flip(mask, axis=1).copy()

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(flip_image)
axes[0].set_title("flipped image")
axes[1].imshow(flip_mask, cmap=cmap, vmin=0, vmax=2)
axes[1].set_title("flipped mask")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 47-3. Resize interpolation 차이

In [ ]:
mask_tensor = torch.from_numpy(mask).unsqueeze(0).unsqueeze(0).float()

nearest = F.interpolate(mask_tensor, size=(48, 48), mode="nearest").squeeze().long()
bilinear = F.interpolate(mask_tensor, size=(48, 48), mode="bilinear", align_corners=False).squeeze()

print("nearest unique values:", torch.unique(nearest).tolist())
print("bilinear min/max:", float(bilinear.min()), float(bilinear.max()))
print("bilinear has fractional values:", bool(((bilinear % 1) != 0).any()))

mask를 bilinear로 줄이면 class id 사이의 소수 값이 생깁니다. segmentation label에는 이런 값이 들어가면 안 됩니다.

## 47-4. 해상도 실험 기준

| 해상도 증가 효과 | 장점 | 비용 |
|---|---|---|
| 작은 객체 보존 | detail과 boundary 개선 가능 | GPU memory 증가 |
| crop size 증가 | 넓은 context 확인 | batch size 감소 |
| test scale 증가 | 예측 품질 개선 가능 | inference latency 증가 |

SegFormer 계열에서는 encoder stage feature shape와 decoder upsample 비율도 함께 확인해야 합니다.

## 정리

- geometric augmentation은 image와 mask에 같은 파라미터로 적용해야 합니다.
- mask resize에는 nearest interpolation을 사용합니다.
- 입력 해상도는 mIoU, memory, latency를 함께 보며 실험해야 합니다.
- 다음 노트북 `48_Inference_Error_Analysis와_결과_시각화.ipynb`에서는 예측 결과를 해석하고 실패 사례를 분석합니다.